In [2]:
# ============================================================
# Stock-level INDMOM (GHZ_20161231_DataSetUp.sas style)
#
# Step 1: For each stock i and month t, compute mom12m (12-2 momentum):
#   mom12m_{i,t} = Π_{k=2..12} (1 + ret_{i,t-k}) - 1
# (i.e., use months t-2 ... t-12; exclude t-1)
#
# Step 2: Industry momentum for (sic2, t):
#   indmom_{sic2,t} = mean_i( mom12m_{i,t} ) within (sic2, t)
#
# Step 3: Stock-level indmom:
#   indmom_{i,t} = indmom_{sic2(i,t), t}
#
# Output: permno, date (calendar month-end), indmom
# Output window: 1956-01 .. 1989-12
# Data source: WRDS CRSP msf + msenames
# ============================================================

import numpy as np
import pandas as pd
import wrds

# -----------------------
# 0) Parameters
# -----------------------
OUT_START = pd.Timestamp("1956-01-31")
OUT_END   = pd.Timestamp("1989-12-31")

# Need history for mom12m at 1956-01 (t-12..t-2); pull earlier as buffer
PULL_START = pd.Timestamp("1954-01-01")
PULL_END   = OUT_END

OUT_CSV     = "indmom_stocklevel_1956_1989.csv"
OUT_PARQUET = "indmom_stocklevel_1956_1989.parquet"

# Optional filters (common in GHZ-style setups)
FILTER_COMMON = True   # shrcd in (10,11)
FILTER_EXCH   = True   # exchcd in (1,2,3)

# -----------------------
# 1) Connect WRDS
# -----------------------
db = wrds.Connection()

# -----------------------
# 2) Pull CRSP monthly returns + identifiers (siccd/shrcd/exchcd) from msenames
# -----------------------
q = f"""
select
  a.permno,
  a.date,
  a.ret,
  b.siccd,
  b.shrcd,
  b.exchcd
from crsp.msf as a
left join crsp.msenames as b
  on a.permno = b.permno
 and b.namedt <= a.date
 and a.date <= b.nameendt
where a.date >= '{PULL_START.date()}'
  and a.date <= '{PULL_END.date()}'
"""
msf = db.raw_sql(q, date_cols=["date"])

# ret sometimes contains non-numeric codes; coerce
msf["ret"] = pd.to_numeric(msf["ret"], errors="coerce")

if FILTER_COMMON:
    msf = msf[msf["shrcd"].isin([10, 11])].copy()

if FILTER_EXCH:
    msf = msf[msf["exchcd"].isin([1, 2, 3])].copy()

# sic2 from siccd
msf["siccd"] = pd.to_numeric(msf["siccd"], errors="coerce")
msf["sic2"] = np.floor(msf["siccd"] / 100.0)

# Calendar month-end label
msf["date_m"] = msf["date"].dt.to_period("M").dt.to_timestamp("M")

# Keep the essential panel keys
msf = msf.sort_values(["permno", "date_m"]).reset_index(drop=True)

# -----------------------
# 3) Compute mom12m (12-2 momentum) per permno-month
#    Mimic SAS behavior: compute lags on non-missing ret
# -----------------------
def add_mom12m(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()

    # Mimic "where not missing(ret)" before lagging
    g = g.dropna(subset=["ret"]).copy()

    # Safety: log1p requires 1+ret > 0
    g = g[(1.0 + g["ret"]) > 0].copy()
    if g.empty:
        return g

    log1p = np.log1p(g["ret"].to_numpy(dtype=float))

    # shift by 2 months, then rolling sum of length 11 => k=2..12
    shifted = pd.Series(log1p, index=g.index).shift(2)
    rollsum = shifted.rolling(window=11, min_periods=11).sum()

    g["mom12m"] = np.expm1(rollsum)
    return g

tmp = (
    msf.groupby("permno", group_keys=False, sort=False)
       .apply(add_mom12m)
)

# -----------------------
# 4) Industry indmom = mean(mom12m) by (sic2, date_m)
# -----------------------
tmp = tmp.dropna(subset=["sic2"]).copy()

industry_indmom = (
    tmp.groupby(["sic2", "date_m"], as_index=False)["mom12m"]
       .mean()  # mean ignores NaN
       .rename(columns={"date_m": "date", "mom12m": "indmom"})
)

# -----------------------
# 5) Merge back to stock-month panel => stock-level indmom (permno, date, indmom)
# -----------------------
panel = msf[["permno", "date_m", "sic2"]].drop_duplicates().rename(columns={"date_m": "date"})
panel = panel.dropna(subset=["sic2"]).copy()

stock_indmom = (
    panel.merge(industry_indmom, on=["sic2", "date"], how="left")
         .loc[(lambda d: (d["date"] >= OUT_START) & (d["date"] <= OUT_END)), ["permno", "date", "indmom"]]
         .sort_values(["date", "permno"])
         .reset_index(drop=True)
)

# -----------------------
# 6) Save + quick checks
# -----------------------
stock_indmom.to_csv(OUT_CSV, index=False)
stock_indmom.to_parquet(OUT_PARQUET, index=False)

print(stock_indmom.head())
print("min date:", stock_indmom["date"].min(), "| max date:", stock_indmom["date"].max())
print("unique months:", stock_indmom["date"].nunique())
print("unique permno:", stock_indmom["permno"].nunique())
print("indmom missing rate:", stock_indmom["indmom"].isna().mean())
print("Saved:", OUT_CSV, "and", OUT_PARQUET)

Enter your WRDS username [zhouzixian]: zixian_zhou
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  n


You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


/var/folders/05/nw_2bk5142l2vdd8kj666hw40000gn/T/ipykernel_97816/2718588683.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_mom12m)


   permno       date    indmom
0   10006 1956-01-31  0.214114
1   10014 1956-01-31  0.258589
2   10022 1956-01-31  0.126291
3   10030 1956-01-31  0.356767
4   10057 1956-01-31  0.200804
min date: 1956-01-31 00:00:00 | max date: 1989-12-31 00:00:00
unique months: 408
unique permno: 13583
indmom missing rate: 0.0008229075135640264
Saved: indmom_stocklevel_1956_1989.csv and indmom_stocklevel_1956_1989.parquet
